In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
from tensorly.decomposition import tucker
import tensorly as tl

tl.set_backend('pytorch')

# ==========================================
# 1. SVDLinear (Supports Fixed & Adaptive)
# ==========================================
class SVDLinear(nn.Module):
    def __init__(self, in_features, out_features, rank=16, adaptive=False):
        super().__init__()
        self.adaptive = adaptive
        
        W = torch.empty(out_features, in_features)
        nn.init.kaiming_uniform_(W, a=np.sqrt(5))
        U, S, Vh = torch.linalg.svd(W, full_matrices=False)
        
        self.rank = min(rank, min(in_features, out_features))
        
        self.U = nn.Parameter(U[:, :self.rank].clone())
        self.S = nn.Parameter(S[:self.rank].clone())
        self.V = nn.Parameter(Vh[:self.rank, :].clone())
        self.bias = nn.Parameter(torch.zeros(out_features))

    def forward(self, x):
        h = x @ self.V.T
        h = h * self.S
        out = h @ self.U.T + self.bias
        return out

    def prune_spectral_entropy(self, threshold=0.9):
        if not self.adaptive or self.rank <= 1:
            return False
            
        sigma = torch.abs(self.S.data)
        p = sigma / torch.sum(sigma)
        
        p_safe = p + 1e-9
        H = -torch.sum(p * torch.log(p_safe))
        
        sorted_p, indices = torch.sort(p, descending=True)
        sorted_info = -(sorted_p * torch.log(sorted_p + 1e-9))
        cum_info = torch.cumsum(sorted_info, dim=0)
        
        k = torch.sum(cum_info < threshold * H).item() + 1
        k = max(1, min(k, self.rank - 1)) # Force at least 1 rank drop if possible
        
        if k < self.rank:
            keep_idx = torch.sort(indices[:k])[0]
            self.S = nn.Parameter(self.S.data[keep_idx])
            self.U = nn.Parameter(self.U.data[:, keep_idx])
            self.V = nn.Parameter(self.V.data[keep_idx, :])
            self.rank = k
            return True
        return False

# ==========================================
# 2. TuckerConv (Supports Fixed & Adaptive)
# ==========================================
class TuckerConv(nn.Module):
    def __init__(self, in_ch, out_ch, ranks=(16, 16), adaptive=False):
        super().__init__()
        self.adaptive = adaptive
        self.r_o, self.r_i = ranks
        self.in_ch = in_ch
        self.out_ch = out_ch
        
        # Initialize standard weight and decompose
        W = torch.empty(out_ch, in_ch, 3, 3)
        nn.init.kaiming_uniform_(W, a=np.sqrt(5))
        core, [U_out, U_in, _, _] = tucker(W.data, rank=[self.r_o, self.r_i, 3, 3], init='svd')
        
        # Build modules
        self.reduce = nn.Conv2d(in_ch, self.r_i, 1, bias=False)
        self.core   = nn.Conv2d(self.r_i, self.r_o, 3, padding=1, bias=False)
        self.expand = nn.Conv2d(self.r_o, out_ch, 1, bias=True)
        
        # Load weights
        self.reduce.weight.data.copy_(U_in.T.unsqueeze(-1).unsqueeze(-1))
        self.core.weight.data.copy_(core)
        self.expand.weight.data.copy_(U_out.unsqueeze(-1).unsqueeze(-1))

    def forward(self, x):
        return self.expand(self.core(self.reduce(x)))

    def prune_spectral_entropy(self, threshold=0.9):
        if not self.adaptive:
            return False
        
        pruned = False
        device = self.core.weight.device
        
        # --- Prune Input Rank (r_i) ---
        W_in = self.reduce.weight.data.view(self.r_i, self.in_ch)
        _, S_in, _ = torch.linalg.svd(W_in, full_matrices=False)
        p_in = S_in / torch.sum(S_in)
        H_in = -torch.sum(p_in * torch.log(p_in + 1e-9))
        
        sorted_p, sort_idx = torch.sort(p_in, descending=True)
        cum_info = torch.cumsum(-(sorted_p * torch.log(sorted_p + 1e-9)), dim=0)
        k_i = torch.sum(cum_info < threshold * H_in).item() + 1
        k_i = max(1, min(k_i, self.r_i - 1))
        
        if k_i < self.r_i:
            # Sort channels by L2 norm to find which to keep
            norms = torch.norm(W_in, dim=1)
            keep_idx_i = torch.sort(torch.topk(norms, k_i)[1])[0]
            
            # Physically rebuild modules with new dimensions
            new_reduce = nn.Conv2d(self.in_ch, k_i, 1, bias=False).to(device)
            new_core = nn.Conv2d(k_i, self.r_o, 3, padding=1, bias=False).to(device)
            
            new_reduce.weight.data.copy_(self.reduce.weight.data[keep_idx_i, :, :, :])
            new_core.weight.data.copy_(self.core.weight.data[:, keep_idx_i, :, :])
            
            self.reduce = new_reduce
            self.core = new_core
            self.r_i = k_i
            pruned = True

        # --- Prune Output Rank (r_o) ---
        W_out = self.expand.weight.data.view(self.out_ch, self.r_o)
        _, S_out, _ = torch.linalg.svd(W_out, full_matrices=False)
        p_out = S_out / torch.sum(S_out)
        H_out = -torch.sum(p_out * torch.log(p_out + 1e-9))
        
        sorted_p, sort_idx = torch.sort(p_out, descending=True)
        cum_info = torch.cumsum(-(sorted_p * torch.log(sorted_p + 1e-9)), dim=0)
        k_o = torch.sum(cum_info < threshold * H_out).item() + 1
        k_o = max(1, min(k_o, self.r_o - 1))
        
        if k_o < self.r_o:
            norms = torch.norm(W_out, dim=0)
            keep_idx_o = torch.sort(torch.topk(norms, k_o)[1])[0]
            
            new_core = nn.Conv2d(self.r_i, k_o, 3, padding=1, bias=False).to(device)
            new_expand = nn.Conv2d(k_o, self.out_ch, 1, bias=True).to(device)
            
            new_core.weight.data.copy_(self.core.weight.data[keep_idx_o, :, :, :])
            new_expand.weight.data.copy_(self.expand.weight.data[:, keep_idx_o, :, :])
            new_expand.bias.data.copy_(self.expand.bias.data)
            
            self.core = new_core
            self.expand = new_expand
            self.r_o = k_o
            pruned = True
            
        return pruned

# ==========================================
# 3. Master Architecture Class
# ==========================================
class CompressionCNN(nn.Module):
    def __init__(self, mode="baseline", conv_ranks=(16,16), fc_rank=64):
        super().__init__()
        self.mode = mode
        
        # Determine behavior based on requested experiment
        use_tucker = mode in ["fixed_all", "fixed_conv_adaptive_fc", "adaptive_all"]
        adapt_conv = mode == "adaptive_all"
        use_svd_fc = mode in ["fixed_all", "fixed_conv_adaptive_fc", "adaptive_all"]
        adapt_fc   = mode in ["fixed_conv_adaptive_fc", "adaptive_all"]

        # Feature Extractor
        self.features = nn.ModuleList()
        
        # Block 1: 1 -> 32
        if use_tucker:
            self.features.append(TuckerConv(1, 32, ranks=conv_ranks, adaptive=adapt_conv))
        else:
            self.features.append(nn.Conv2d(1, 32, 3, padding=1))
        self.features.extend([nn.BatchNorm2d(32), nn.ReLU(inplace=True)])
        
        # Block 2: 32 -> 64
        if use_tucker:
            self.features.append(TuckerConv(32, 64, ranks=conv_ranks, adaptive=adapt_conv))
        else:
            self.features.append(nn.Conv2d(32, 64, 3, padding=1))
        self.features.extend([nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2)])
        
        # Classifier
        if use_svd_fc:
            self.fc1 = SVDLinear(64*14*14, 128, rank=fc_rank, adaptive=adapt_fc)
            self.bn1 = nn.BatchNorm1d(128)
            self.fc2 = SVDLinear(128, 10, rank=fc_rank, adaptive=adapt_fc)
        else:
            self.fc1 = nn.Linear(64*14*14, 128)
            self.bn1 = nn.BatchNorm1d(128)
            self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        for layer in self.features:
            x = layer(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.bn1(self.fc1(x)))
        return self.fc2(x)

    def trigger_pruning(self, threshold=0.9):
        """Sweeps through all modules and triggers pruning if adaptive is enabled."""
        changed = False
        for module in self.modules():
            if isinstance(module, (SVDLinear, TuckerConv)):
                if module.prune_spectral_entropy(threshold):
                    changed = True
        return changed

    def get_ranks(self):
        """Returns current ranks for tracking."""
        r_c1, r_c2, r_f1, r_f2 = "N/A", "N/A", "N/A", "N/A"
        if hasattr(self.features[0], 'r_o'):
            r_c1 = f"({self.features[0].r_o},{self.features[0].r_i})"
            r_c2 = f"({self.features[3].r_o},{self.features[3].r_i})"
        if hasattr(self.fc1, 'rank'):
            r_f1 = str(self.fc1.rank)
            r_f2 = str(self.fc2.rank)
        return f"C1:{r_c1} C2:{r_c2} F1:{r_f1} F2:{r_f2}"

# ==========================================
# 4. Global Runner
# ==========================================
def run_experiment(mode, epochs=20, batch_size=128):
    print(f"\n{'='*90}")
    print(f" EXPERIMENT: {mode.upper()}")
    print(f"{'='*90}")
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
    train_loader = DataLoader(datasets.MNIST('.', train=True, download=True, transform=transform), batch_size=batch_size, shuffle=True)
    test_loader  = DataLoader(datasets.MNIST('.', train=False, transform=transform), batch_size=1000, shuffle=False)

    # Initializing based on Table 4 setup
    model = CompressionCNN(mode=mode, conv_ranks=(29, 29), fc_rank=115).to(device)
    
    lr, momentum, weight_decay = 1e-3, 0.9, 1e-4
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()
    
    pruning_epochs = [int(epochs * f) for f in np.arange(0.1, 0.9, 0.1)]
    
    # Calculate baseline parameters dynamically for comparison
    dummy_base = CompressionCNN(mode="baseline")
    base_params = sum(p.numel() for p in dummy_base.parameters())
    
    print(f"Epoch | Train Loss | Test Acc(%) | Params    | Compression | Ranks")
    print("-" * 90)

    for ep in range(1, epochs+1):
        
        # 1. Pruning Phase (0.1E -> 0.8E)
        if ep in pruning_epochs:
            if model.trigger_pruning(threshold=0.90):
                optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)

        # 2. Train Phase
        model.train()
        train_loss = 0
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            out = model(data)
            loss = criterion(out, target)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * data.size(0)
        train_loss /= len(train_loader.dataset)

        # 3. Eval Phase
        model.eval()
        correct = 0
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                pred = model(data).argmax(dim=1, keepdim=True)
                correct += pred.eq(target.view_as(pred)).sum().item()
        acc = 100. * correct / len(test_loader.dataset)

        # 4. Logging
        current_params = sum(p.numel() for p in model.parameters())
        compression = 100. * (base_params - current_params) / base_params
        
        print(f"{ep:5d} | {train_loss:10.4f} | {acc:11.2f} | {current_params:9d} | {compression:10.2f}% | {model.get_ranks()}")

if __name__ == '__main__':
    # 1. Full parameter FC and Conv
    run_experiment("baseline", epochs=20)
    
    # 2. Fixed rank FC and Conv
    run_experiment("fixed_all", epochs=20)
    
    # 3. Fixed rank Conv, dynamic spectral FC
    run_experiment("fixed_conv_adaptive_fc", epochs=20)
    
    # 4. Dynamic determining FC and Conv
    run_experiment("adaptive_all", epochs=20)


 EXPERIMENT: BASELINE
Epoch | Train Loss | Test Acc(%) | Params    | Compression | Ranks
------------------------------------------------------------------------------------------
    1 |     0.3423 |       97.88 |   1626314 |       0.00% | C1:N/A C2:N/A F1:N/A F2:N/A
    2 |     0.1028 |       98.51 |   1626314 |       0.00% | C1:N/A C2:N/A F1:N/A F2:N/A
    3 |     0.0692 |       98.70 |   1626314 |       0.00% | C1:N/A C2:N/A F1:N/A F2:N/A
    4 |     0.0527 |       98.78 |   1626314 |       0.00% | C1:N/A C2:N/A F1:N/A F2:N/A
    5 |     0.0424 |       98.83 |   1626314 |       0.00% | C1:N/A C2:N/A F1:N/A F2:N/A
    6 |     0.0352 |       98.82 |   1626314 |       0.00% | C1:N/A C2:N/A F1:N/A F2:N/A
    7 |     0.0292 |       98.97 |   1626314 |       0.00% | C1:N/A C2:N/A F1:N/A F2:N/A
    8 |     0.0244 |       98.86 |   1626314 |       0.00% | C1:N/A C2:N/A F1:N/A F2:N/A
    9 |     0.0209 |       98.91 |   1626314 |       0.00% | C1:N/A C2:N/A F1:N/A F2:N/A
   10 |     0.0178